# Tuned against untuned: is the search worth it?

One row is one client the bank called during a deposit campaign; `y` is `yes` when the client subscribed. The campaign lead: "A colleague spent a day tuning last year's model. Was that a good use of a day?"

## The starter

The cell applies D01 5.5's decision (`duration` is dropped: it is known only after the call), splits 80 / 20 with `stratify=y, random_state=42`, and defines `pre`, a `ColumnTransformer` that one-hot encodes the text columns and passes the numbers through. Expect 36,168 training and 9,043 test clients, 11.7% subscribers.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("data/bank_marketing.csv")
df = df.rename(columns={"deposit": "y", "day": "day_of_week"})    # one set of names for every download route
for col in ["job", "education", "contact", "poutcome"]:
    df[col] = df[col].fillna("unknown")
y = (df["y"] == "yes").astype(int)
X = df.drop(columns=["y", "duration"])                             # duration is known only after the call
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

num_cols = list(X.select_dtypes(include="number").columns)
cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)
pre = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
print(len(X_train), "training clients,", len(X_test), "test clients,", f"{y.mean():.3f} subscribed")

## Step 1 · Three folds

`cv = StratifiedKFold(3, shuffle=True, random_state=42)`. Three folds keep the search short on the course VM; the demonstration used five.

## Step 2 · The untuned boosted model

`cross_val_score(make_pipeline(pre, HistGradientBoostingClassifier(random_state=42)), X_train, y_train, cv=cv, scoring="roc_auc").mean()`.

## Step 3 · A random search with 15 candidates

```python
P = "histgradientboostingclassifier__"
dist = {P + "learning_rate": loguniform(0.01, 0.5), P + "max_leaf_nodes": randint(7, 128),
        P + "min_samples_leaf": randint(5, 200), P + "l2_regularization": loguniform(1e-3, 10)}
search = RandomizedSearchCV(model, dist, n_iter=15, cv=cv, scoring="roc_auc",
                            random_state=42, n_jobs=-1).fit(X_train, y_train)
```

Print `best_params_` and `best_score_`. How many models did the search fit?

## Step 4 · How wide is the range?

Sort `pd.DataFrame(search.cv_results_)` by `rank_test_score` and look at the best and worst `mean_test_score`.

## Step 5 · The test clients, once per model

Score the test clients with the defaults fitted on all training rows, and with `search.best_estimator_`.

## Step 6 · The same question for a decision tree

`DecisionTreeClassifier(random_state=42)`: its 5-fold CV score with the defaults, then a `GridSearchCV` over `max_depth` in `[3, 4, 5, 6, 8, 10]` and `min_samples_leaf` in `[20, 50, 100, 200]`. Use five folds here, as the demonstration did: `StratifiedKFold(5, shuffle=True, random_state=42)`.

## Step 7 · "Was the day of tuning worth it?"

For each model, the gain from tuning in validation ROC AUC, set against the spread across folds. Two sentences, one per model, with the numbers.

## Check yourself